# DNABERT2 Alpine HPC Benchmark

This notebook is the clean Alpine/CURC path for running the full DNABERT2 benchmark.

Use it after the local CPU smoke test has shown that the SeqTrainer DNABERT2 pipeline can write artifacts. The purpose here is the **real comparable run** against CNN-v2.

Scientific constants that must stay fixed:

- same train/validation/test CSV files as CNN-v2
- seed `42`
- threshold selected on validation MCC only
- test split used only for final reporting
- primary metric: test MCC
- secondary metric: test AUPRC
- artifact format: `metrics.csv`, `metrics.json`, `predictions.csv`, `manifest.json`, `history.csv`, `checkpoints/`, `embeddings/`


## 1. Why Alpine

Your local machine does not expose an NVIDIA CUDA GPU to PyTorch. DNABERT2 is much heavier than CNN and should be benchmarked on a real GPU node.

For Alpine, prefer NVIDIA partitions:

- `aa100`: NVIDIA A100, recommended first choice
- `al40`: NVIDIA L40, also viable

Avoid `ami100` for this current SeqTrainer DNABERT2 path unless we intentionally port the workflow to ROCm, because the current implementation and previous debugging assume PyTorch CUDA/NVIDIA behavior.

This notebook writes SLURM scripts that request one GPU using `--gres=gpu:1`.


## 2. One-time setup on Alpine login node

Run this once after cloning the branch on Alpine.

```bash
git clone --branch issue-3-all-model-baselines https://github.com/simplyshree/SeqTrainer.git
cd SeqTrainer

python -m venv .venv-dnabert2
source .venv-dnabert2/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install -e ".[torch]"
python -m pip install "transformers==4.29.2" "einops>=0.6" "accelerate>=0.20"
```

If CURC recommends specific Python/CUDA modules, load those before creating the venv. Keep the same environment for smoke and full jobs.


## 3. Required data files

The full DNABERT2 run must use the same split files as CNN-v2:

```text
data/promoter_classification/train_EP_DNA_BERT2_genomic_order.csv
data/promoter_classification/eval_EP_DNA_BERT2_genomic_order.csv
data/promoter_classification/test_EP_DNA_BERT2_genomic_order.csv
```

If they are not extracted yet, extract them from:

```text
data/data_DNABERT/promoter_classification_DNABERT.zip
```


In [1]:
from pathlib import Path
import json
import os


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config-examples" / "benchmarks" / "dnabert2_frozen.toml").exists():
            return candidate
    raise RuntimeError("Could not find SeqTrainer repo root. Open this notebook from inside the SeqTrainer checkout.")

REPO_DIR = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_DIR)

print("Repo:", REPO_DIR)
print("Frozen config:", REPO_DIR / "config-examples" / "benchmarks" / "dnabert2_frozen.toml")
print("Smoke config:", REPO_DIR / "config-examples" / "benchmarks" / "dnabert2_smoke.toml")


Repo: C:\Users\Sgoff\MYfile\Desktop\PYThh\SeqTrainer
Frozen config: C:\Users\Sgoff\MYfile\Desktop\PYThh\SeqTrainer\config-examples\benchmarks\dnabert2_frozen.toml
Smoke config: C:\Users\Sgoff\MYfile\Desktop\PYThh\SeqTrainer\config-examples\benchmarks\dnabert2_smoke.toml


## 4. Write Alpine SLURM scripts

Edit these placeholders after the files are written:

- `<YOUR_ACCOUNT>`: your CURC allocation/account
- `#SBATCH --partition=aa100`: change to `al40` if using L40
- module lines: uncomment/load the Python/CUDA modules recommended for your Alpine account

Run the smoke script first. If it succeeds, run the full frozen benchmark script.


In [2]:
SLURM_DIR = REPO_DIR / "notebooks" / "benchmarks_sg" / "dnabert_benchmark"
LOG_DIR = REPO_DIR / "logs"
SLURM_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

smoke_script = """#!/bin/bash
#SBATCH --job-name=seqtrainer-dnabert2-smoke
#SBATCH --partition=aa100
#SBATCH --qos=normal
#SBATCH --account=<YOUR_ACCOUNT>
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=64G
#SBATCH --time=00:30:00
#SBATCH --output=logs/%x-%j.out
#SBATCH --error=logs/%x-%j.err

set -euo pipefail

cd "$SLURM_SUBMIT_DIR"
mkdir -p logs

# Load CURC-recommended modules if required for your environment.
# module purge
# module load python
# module load cuda

source .venv-dnabert2/bin/activate
export PYTHONPATH="$PWD/src"
export TOKENIZERS_PARALLELISM=false

python - <<'PY'
import torch, transformers, seqtrainer
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
print('cuda_device_count', torch.cuda.device_count())
if torch.cuda.is_available():
    print('cuda_device_name', torch.cuda.get_device_name(0))
print('transformers', transformers.__version__)
print('seqtrainer', seqtrainer.__file__)
PY

python -m seqtrainer.cli.main benchmark prepare-dnabert2 \
  config-examples/benchmarks/dnabert2_smoke.toml \
  --base-dir . \
  --output-dir outputs/benchmarks/dnabert2_alpine_tokenization_smoke
"""

frozen_script = """#!/bin/bash
#SBATCH --job-name=seqtrainer-dnabert2-frozen
#SBATCH --partition=aa100
#SBATCH --qos=normal
#SBATCH --account=<YOUR_ACCOUNT>
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=96G
#SBATCH --time=12:00:00
#SBATCH --output=logs/%x-%j.out
#SBATCH --error=logs/%x-%j.err

set -euo pipefail

cd "$SLURM_SUBMIT_DIR"
mkdir -p logs

# Load CURC-recommended modules if required for your environment.
# module purge
# module load python
# module load cuda

source .venv-dnabert2/bin/activate
export PYTHONPATH="$PWD/src"
export TOKENIZERS_PARALLELISM=false
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

python - <<'PY'
import torch, transformers, seqtrainer
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
print('cuda_device_count', torch.cuda.device_count())
if torch.cuda.is_available():
    print('cuda_device_name', torch.cuda.get_device_name(0))
print('transformers', transformers.__version__)
print('seqtrainer', seqtrainer.__file__)
PY

python -m seqtrainer.cli.main benchmark run \
  config-examples/benchmarks/dnabert2_frozen.toml \
  --base-dir . \
  --strict
"""

smoke_path = SLURM_DIR / "alpine_dnabert2_smoke.sbatch"
frozen_path = SLURM_DIR / "alpine_dnabert2_frozen.sbatch"
smoke_path.write_text(smoke_script, encoding="utf-8")
frozen_path.write_text(frozen_script, encoding="utf-8")

print("Wrote", smoke_path)
print("Wrote", frozen_path)


Wrote C:\Users\Sgoff\MYfile\Desktop\PYThh\SeqTrainer\notebooks\benchmarks_sg\dnabert_benchmark\alpine_dnabert2_smoke.sbatch
Wrote C:\Users\Sgoff\MYfile\Desktop\PYThh\SeqTrainer\notebooks\benchmarks_sg\dnabert_benchmark\alpine_dnabert2_frozen.sbatch


## 5. Submit jobs on Alpine

From the SeqTrainer repo root on Alpine:

```bash
sbatch notebooks/benchmarks_sg/dnabert_benchmark/alpine_dnabert2_smoke.sbatch
```

Check the log in `logs/`. The smoke log must show:

```text
cuda_available True
```

Then submit the full frozen benchmark:

```bash
sbatch notebooks/benchmarks_sg/dnabert_benchmark/alpine_dnabert2_frozen.sbatch
```

If `aa100` is busy or unavailable, change `#SBATCH --partition=aa100` to `#SBATCH --partition=al40` and resubmit.


## 6. Inspect completed full benchmark

After the full job finishes, check:

```bash
ls outputs/benchmarks/dnabert2_frozen_ep_genomic_order
cat outputs/benchmarks/dnabert2_frozen_ep_genomic_order/metrics.csv
```

Expected artifacts:

```text
metrics.csv
metrics.json
predictions.csv
manifest.json
history.csv
checkpoints/best_model.pt
embeddings/train_embeddings.pt
embeddings/validation_embeddings.pt
embeddings/test_embeddings.pt
```


In [3]:
from pathlib import Path
import pandas as pd

OUT = REPO_DIR / "outputs" / "benchmarks" / "dnabert2_frozen_ep_genomic_order"
metrics_path = OUT / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)
else:
    print("Full Alpine DNABERT2 metrics are not present yet.")
    print("Run the Alpine frozen SLURM job first, then rerun this cell.")


Full Alpine DNABERT2 metrics are not present yet.
Run the Alpine frozen SLURM job first, then rerun this cell.


## 7. Compare against CNN-v2

After the full DNABERT2 run exists, compare it with CNN-v2:

```bash
seqtrainer benchmark compare   outputs/benchmarks/cnn_v2_regularized_ep_genomic_order   outputs/benchmarks/dnabert2_frozen_ep_genomic_order   --output-dir outputs/benchmarks/comparison_cnn_v2_dnabert2
```

Remember: only the full DNABERT2 run on the complete shared split is comparable to CNN-v2. The local CPU smoke result is only for pipeline validation.
